<a href="https://colab.research.google.com/github/aferenaz/MLOPS2-Mini-TP7/blob/main/mini_tp7_grpc_gout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-TP 7 · Seguridad con SAIF — gRPC

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por gRPC**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** acceso no autenticado al servicio y *man-in-the-middle* (canal sin cifrar).  
**Controles:** token en la *metadata* (interceptor) · TLS/mTLS · límite de tamaño de mensaje.  
**Elementos SAIF:** bases sólidas · coherencia de controles.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.


> **Caso elegido: pronóstico de demanda de Gout** (cadena de panaderías sin gluten, 9 sucursales y 96 SKUs).
> Se elige gRPC porque el TP final ya expone el modelo por gRPC; este control queda incorporado al servicio.

### Dependencias
```bash
uv add grpcio grpcio-tools
```

In [ ]:
# Instalación (con uv desde la raíz del curso): uv add grpcio grpcio-tools pyjwt scikit-learn numpy joblib pandas cryptography
# En Colab, descomentar:
# %pip install -q grpcio grpcio-tools pyjwt scikit-learn joblib cryptography

## 1. Definir el servicio (proto) y compilarlo

In [ ]:
proto = '''syntax = "proto3";
package modelo;
service Modelo { rpc Predecir (Entrada) returns (Salida); }
message Entrada { repeated float features = 1; }
message Salida { int32 prediccion = 1; float confianza = 2; }
'''
open("modelo.proto", "w").write(proto)
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "grpc_tools.protoc", "-I.",
      "--python_out=.", "--grpc_python_out=.", "modelo.proto"],
      capture_output=True, text=True).returncode, "= 0 si compiló")

0 = 0 si compiló


## 2. Control principal (GUIADO) — interceptor de autenticación + límite de tamaño

In [ ]:
import sys; sys.path.insert(0, ".")
import grpc
from concurrent import futures
import modelo_pb2 as pb, modelo_pb2_grpc as pbg

TOKEN = "Bearer tok-valido"     # en producción: un JWT verificado, desde el entorno

class AuthInterceptor(grpc.ServerInterceptor):
    def intercept_service(self, continuation, hcd):
        md = dict(hcd.invocation_metadata or [])
        if md.get("authorization") != TOKEN:
            return grpc.unary_unary_rpc_method_handler(
                lambda req, ctx: ctx.abort(grpc.StatusCode.UNAUTHENTICATED, "token inválido"))
        return continuation(hcd)

class Servicio(pbg.ModeloServicer):
    def Predecir(self, request, context):
        return pb.Salida(prediccion=int(sum(request.features) > 0), confianza=0.9)

servidor = grpc.server(
    futures.ThreadPoolExecutor(max_workers=2),
    interceptors=[AuthInterceptor()],
    options=[("grpc.max_receive_message_length", 1024 * 1024)])  # límite de 1 MB (anti-DoS)
pbg.add_ModeloServicer_to_server(Servicio(), servidor)
port = servidor.add_insecure_port("localhost:0"); servidor.start()

stub = pbg.ModeloStub(grpc.insecure_channel(f"localhost:{port}"))
try:
    stub.Predecir(pb.Entrada(features=[1.0, 2.0]))
except grpc.RpcError as e:
    print("sin token:", e.code().name)
r = stub.Predecir(pb.Entrada(features=[1.0, 2.0]), metadata=[("authorization", TOKEN)])
print("con token:", r.prediccion, round(r.confianza, 2))
servidor.stop(0)

sin token: UNAUTHENTICATED
con token: 1 0.9


<threading.Event at 0x7f24890db3b0: unset>

## 3. Entrega # TODO` sobre el modelo de Gout

El servicio expone el **pronóstico de demanda de Gout** (unidades por sucursal, SKU y día).
Es el mismo servicio gRPC del TP final, ahora asegurado.

| TODO | Qué se hace acá |
|---|---|
| 1 | Interceptor que verifica un **JWT** (firma, vencimiento, emisor) y autoriza por **rol** |
| 2 | El servicio carga **el modelo de Gout** y valida la entrada |
| 3 | Paso de `insecure_port` a **TLS** (documentado y demostrado con un certificado de prueba) |
| Gobernanza | **Logging de auditoría** en JSON + **model card** |

### Configuración del modelo

Poné el modelo entrenado de Gout en `MODELO_PATH` (un `joblib` con el estimador, por ejemplo el XGBoost de la memoria)
y la lista de features **en el mismo orden con que se entrenó**. Si el archivo no está, el notebook entrena un
modelo **de reemplazo** sobre datos sintéticos con la misma forma, solo para que corra de punta a punta.


In [ ]:
import os, json, time, hashlib, logging
from pathlib import Path
import numpy as np
import joblib

MODELO_PATH = Path(os.environ.get("GOUT_MODELO_PATH", "modelo_gout.joblib"))

# Orden exacto de las features con que se entrenó el modelo (ajustar al modelo real)
FEATURES = [
    "sucursal_cod",     # sucursal anonimizada, codificada
    "sku_cod",          # SKU codificado
    "dia_semana",       # 0 = lunes ... 6 = domingo
    "mes",              # 1 a 12
    "es_feriado",       # 0 / 1
    "lag_1",            # unidades vendidas el día anterior
    "lag_7",            # unidades vendidas hace 7 días
    "media_movil_7",    # promedio de los últimos 7 días
]
N_FEATURES = len(FEATURES)

def entrenar_reemplazo(path):
    """Modelo de REEMPLAZO sobre datos sintéticos con la forma de Gout. No usar en la entrega."""
    from sklearn.ensemble import HistGradientBoostingRegressor
    rng = np.random.default_rng(42)
    n = 5000
    X = np.column_stack([
        rng.integers(0, 9, n),       # 9 sucursales
        rng.integers(0, 96, n),      # 96 SKUs
        rng.integers(0, 7, n),
        rng.integers(1, 13, n),
        rng.integers(0, 2, n),
        rng.poisson(20, n),
        rng.poisson(20, n),
        rng.normal(20, 4, n),
    ]).astype(float)
    y = 0.4 * X[:, 5] + 0.3 * X[:, 6] + 0.3 * X[:, 7] + 3 * (X[:, 2] >= 5) - 4 * X[:, 4] + rng.normal(0, 2, n)
    modelo = HistGradientBoostingRegressor(random_state=42).fit(X, np.clip(y, 0, None))
    joblib.dump({"modelo": modelo, "features": FEATURES, "version": "reemplazo-sintetico"}, path)

if not MODELO_PATH.exists():
    print(f"No se encontró {MODELO_PATH}: se entrena un modelo de REEMPLAZO (solo para la demo).")
    entrenar_reemplazo(MODELO_PATH)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

MODELO_SHA256 = sha256(MODELO_PATH)
# Integridad: si se publica el hash esperado (p. ej. desde MLflow), se rechaza un artefacto distinto
SHA_ESPERADO = os.environ.get("GOUT_MODELO_SHA256")
if SHA_ESPERADO and SHA_ESPERADO != MODELO_SHA256:
    raise RuntimeError("El modelo no coincide con el hash publicado: no se sirve")
artefacto = joblib.load(MODELO_PATH)
if isinstance(artefacto, dict):           # formato {"modelo", "features", "version"}
    MODELO = artefacto["modelo"]
    MODELO_VERSION = artefacto.get("version", MODELO_PATH.stem)
    assert list(artefacto.get("features", FEATURES)) == FEATURES, "Las features no coinciden con las del entrenamiento"
else:                                     # estimador suelto (p. ej. XGBRegressor)
    MODELO, MODELO_VERSION = artefacto, MODELO_PATH.stem
print("Modelo:", type(MODELO).__name__, "| versión:", MODELO_VERSION, "| sha256:", MODELO_SHA256[:16], "...")

Modelo: HistGradientBoostingRegressor | versión: reemplazo-sintetico | sha256: cec467bd76692a18 ...


### Servicio gRPC de Gout

El mensaje de entrada lleva la sucursal y el SKU (para auditoría) y el vector de features.
La salida es la demanda estimada en unidades y la versión del modelo que respondió.

In [ ]:
proto_gout = '''syntax = "proto3";
package gout;
service Demanda { rpc Predecir (Consulta) returns (Pronostico); }
message Consulta {
  string sucursal = 1;          // id anonimizado, p. ej. "S03"
  string sku = 2;               // p. ej. "SKU-0412"
  repeated float features = 3;  // en el orden de FEATURES
}
message Pronostico {
  float unidades = 1;
  string modelo_version = 2;
}
'''
Path("gout.proto").write_text(proto_gout)
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "grpc_tools.protoc", "-I.", "--python_out=.",
                    "--grpc_python_out=.", "gout.proto"], capture_output=True, text=True)
print(r.returncode, "= 0 si compiló", r.stderr)

0 = 0 si compiló 


### Gobernanza: logging de auditoría

Cada llamada deja una línea JSON en `auditoria_gout.jsonl`: quién (`sub`, rol), qué método, sucursal y SKU,
resultado (código gRPC) y latencia. **No se guardan las features crudas**, solo su hash, para poder
rastrear una consulta sin exponer datos comerciales de la cadena.

In [ ]:
AUDIT_PATH = Path("auditoria_gout.jsonl")
AUDIT_PATH.unlink(missing_ok=True)

auditor = logging.getLogger("gout.auditoria")
auditor.setLevel(logging.INFO)
auditor.handlers.clear()
_h = logging.FileHandler(AUDIT_PATH, encoding="utf-8")
_h.setFormatter(logging.Formatter("%(message)s"))
auditor.addHandler(_h)
auditor.propagate = False

def auditar(**evento):
    evento = {"ts": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "modelo_version": MODELO_VERSION, **evento}
    auditor.info(json.dumps(evento, ensure_ascii=False))

### TODO 1 — Interceptor con JWT y autorización por rol

- El secreto sale del **entorno** (`GOUT_JWT_SECRET`); el valor por defecto es solo para la demo.
- Se verifica firma (HS256), vencimiento (`exp`), emisor (`iss`) y que el token tenga `sub` y `rol`.
- Cada método declara qué roles puede usarlo. Token ausente o inválido → `UNAUTHENTICATED`; rol sin permiso → `PERMISSION_DENIED`.
- El mismo interceptor registra la auditoría de **todas** las llamadas, también las rechazadas.

In [ ]:
import grpc, jwt
from concurrent import futures
import gout_pb2 as gpb, gout_pb2_grpc as gpbg

JWT_SECRET = os.environ.get("GOUT_JWT_SECRET", "solo-demo-cambiar-en-produccion-por-secreto-de-32+bytes")
JWT_ISSUER = "gout-mlops"
METODO_PREDECIR = "/gout.Demanda/Predecir"
ROLES_POR_METODO = {METODO_PREDECIR: {"planificador", "sistema"}}   # "invitado" no puede pronosticar

def emitir_token(sub, rol, minutos=15):
    """Lo usa el emisor de identidad; acá sirve para las pruebas."""
    ahora = int(time.time())
    return jwt.encode({"sub": sub, "rol": rol, "iss": JWT_ISSUER, "iat": ahora,
                       "exp": ahora + minutos * 60}, JWT_SECRET, algorithm="HS256")

def _rechazo(codigo, mensaje):
    def handler(request, context):
        context.abort(codigo, mensaje)
    return grpc.unary_unary_rpc_method_handler(handler)

class JWTAuthInterceptor(grpc.ServerInterceptor):
    def intercept_service(self, continuation, hcd):
        metodo = hcd.method
        md = dict(hcd.invocation_metadata or [])
        auth = md.get("authorization", "")
        try:
            if not auth.startswith("Bearer "):
                raise jwt.InvalidTokenError("falta el esquema Bearer")
            claims = jwt.decode(auth[7:], JWT_SECRET, algorithms=["HS256"], issuer=JWT_ISSUER,
                                options={"require": ["exp", "iss", "sub"]})
        except jwt.ExpiredSignatureError:
            auditar(metodo=metodo, sub=None, rol=None, resultado="UNAUTHENTICATED", motivo="token vencido")
            return _rechazo(grpc.StatusCode.UNAUTHENTICATED, "token vencido")
        except jwt.InvalidTokenError as e:
            auditar(metodo=metodo, sub=None, rol=None, resultado="UNAUTHENTICATED", motivo=str(e))
            return _rechazo(grpc.StatusCode.UNAUTHENTICATED, "token inválido")

        sub, rol = claims.get("sub"), claims.get("rol")
        if rol not in ROLES_POR_METODO.get(metodo, set()):
            auditar(metodo=metodo, sub=sub, rol=rol, resultado="PERMISSION_DENIED", motivo="rol sin permiso")
            return _rechazo(grpc.StatusCode.PERMISSION_DENIED, f"el rol '{rol}' no puede llamar a {metodo}")

        handler = continuation(hcd)
        if handler is None or handler.unary_unary is None:
            return handler
        interno = handler.unary_unary

        def con_auditoria(request, context):
            t0 = time.perf_counter()
            evento = {"metodo": metodo, "sub": sub, "rol": rol,
                      "sucursal": getattr(request, "sucursal", None), "sku": getattr(request, "sku", None),
                      "features_sha256": hashlib.sha256(np.asarray(request.features, dtype=np.float32).tobytes()).hexdigest()[:16]}
            try:
                respuesta = interno(request, context)
                auditar(**evento, resultado="OK", latencia_ms=round((time.perf_counter() - t0) * 1000, 2))
                return respuesta
            except Exception:
                codigo = context.code() if hasattr(context, "code") else None
                auditar(**evento, resultado=getattr(codigo, "name", "ERROR"),
                        latencia_ms=round((time.perf_counter() - t0) * 1000, 2))
                raise

        return grpc.unary_unary_rpc_method_handler(
            con_auditoria,
            request_deserializer=handler.request_deserializer,
            response_serializer=handler.response_serializer)

### TODO 2 — El servicio usa el modelo de Gout

Además de la autenticación, el servicio **valida la entrada** antes de llamar al modelo:
cantidad de features, valores finitos y no negativos en las ventas. Una entrada mal formada devuelve
`INVALID_ARGUMENT` y nunca llega al modelo.

In [ ]:
IDX_NO_NEGATIVAS = [FEATURES.index(f) for f in ("lag_1", "lag_7", "media_movil_7") if f in FEATURES]

class ServicioDemanda(gpbg.DemandaServicer):
    def Predecir(self, request, context):
        x = np.asarray(request.features, dtype=float)
        if x.shape != (N_FEATURES,):
            context.abort(grpc.StatusCode.INVALID_ARGUMENT,
                          f"se esperaban {N_FEATURES} features ({', '.join(FEATURES)}), llegaron {x.size}")
        if not np.all(np.isfinite(x)) or np.any(x[IDX_NO_NEGATIVAS] < 0):
            context.abort(grpc.StatusCode.INVALID_ARGUMENT, "features no finitas o ventas negativas")
        unidades = float(max(0.0, MODELO.predict(x.reshape(1, -1))[0]))
        return gpb.Pronostico(unidades=unidades, modelo_version=MODELO_VERSION)

MAX_MSG = 64 * 1024   # 64 KB: una consulta legítima pesa menos de 100 bytes

def crear_servidor():
    servidor = grpc.server(
        futures.ThreadPoolExecutor(max_workers=4),
        interceptors=[JWTAuthInterceptor()],
        options=[("grpc.max_receive_message_length", MAX_MSG),
                 ("grpc.max_send_message_length", MAX_MSG)])
    gpbg.add_DemandaServicer_to_server(ServicioDemanda(), servidor)
    return servidor

### Prueba de punta a punta

Se prueban los casos que tiene que resolver el control: sin token, token vencido, token falsificado,
rol sin permiso, entrada inválida y mensaje gigante, más dos llamadas legítimas (un planificador y el DAG de Airflow).

In [ ]:
servidor = crear_servidor()
port = servidor.add_insecure_port("localhost:0")
servidor.start()
canal = grpc.insecure_channel(f"localhost:{port}")
stub = gpbg.DemandaStub(canal)

consulta = gpb.Consulta(sucursal="S03", sku="SKU-0412",
                        features=[3, 412 % 96, 5, 10, 0, 22, 25, 23.4])

def probar(nombre, metadata=None, req=consulta):
    try:
        r = stub.Predecir(req, metadata=metadata or [])
        print(f"{nombre:<28} OK  -> {r.unidades:.1f} unidades (modelo {r.modelo_version})")
        return "OK"
    except grpc.RpcError as e:
        print(f"{nombre:<28} {e.code().name:<20} {e.details()}")
        return e.code().name

def bearer(token):
    return [("authorization", f"Bearer {token}")]

token_falso = jwt.encode({"sub": "x", "rol": "sistema", "iss": JWT_ISSUER,
                          "exp": int(time.time()) + 600}, "secreto-de-un-atacante-que-no-es-el-nuestro-123", algorithm="HS256")
resultados = {
    "sin token": probar("sin token"),
    "token vencido": probar("token vencido", bearer(emitir_token("ana", "planificador", minutos=-1))),
    "token falsificado": probar("token falsificado", bearer(token_falso)),
    "rol invitado": probar("rol invitado", bearer(emitir_token("visita", "invitado"))),
    "features incompletas": probar("features incompletas", bearer(emitir_token("ana", "planificador")),
                                   gpb.Consulta(sucursal="S03", sku="SKU-0412", features=[1, 2, 3])),
    "mensaje gigante (DoS)": probar("mensaje gigante (DoS)", bearer(emitir_token("ana", "planificador")),
                                    gpb.Consulta(sucursal="S03", sku="SKU-0412", features=[1.0] * 50_000)),
    "planificador legítimo": probar("planificador legítimo", bearer(emitir_token("ana", "planificador"))),
    "sistema (Airflow)": probar("sistema (Airflow)", bearer(emitir_token("airflow-dag", "sistema"))),
}
servidor.stop(0)

esperado = {"sin token": "UNAUTHENTICATED", "token vencido": "UNAUTHENTICATED",
            "token falsificado": "UNAUTHENTICATED", "rol invitado": "PERMISSION_DENIED",
            "features incompletas": "INVALID_ARGUMENT", "mensaje gigante (DoS)": "RESOURCE_EXHAUSTED",
            "planificador legítimo": "OK", "sistema (Airflow)": "OK"}
assert resultados == esperado, resultados
print("\nTodos los casos se comportan como se espera.")

sin token                    UNAUTHENTICATED      token inválido
token vencido                UNAUTHENTICATED      token vencido
token falsificado            UNAUTHENTICATED      token inválido
rol invitado                 PERMISSION_DENIED    el rol 'invitado' no puede llamar a /gout.Demanda/Predecir
features incompletas         INVALID_ARGUMENT     se esperaban 8 features (sucursal_cod, sku_cod, dia_semana, mes, es_feriado, lag_1, lag_7, media_movil_7), llegaron 3


mensaje gigante (DoS)        RESOURCE_EXHAUSTED   SERVER: Received message larger than max (200019 vs. 65536)
planificador legítimo        OK  -> 25.7 unidades (modelo reemplazo-sintetico)
sistema (Airflow)            OK  -> 25.7 unidades (modelo reemplazo-sintetico)

Todos los casos se comportan como se espera.


### Auditoría registrada

El mensaje gigante no aparece: gRPC lo descarta por tamaño **antes** de pasar por el interceptor.
Es justamente el objetivo del límite: el servidor ni siquiera deserializa el mensaje.

In [ ]:
import pandas as pd
pd.DataFrame([json.loads(l) for l in AUDIT_PATH.read_text(encoding="utf-8").splitlines()])

,ts,modelo_version,metodo,sub,rol,resultado,motivo,sucursal,sku,features_sha256,latencia_ms
0,2026-10-09T18:21:21+0000,reemplazo-sintetico,/gout.Demanda/Predecir,NaN,NaN,UNAUTHENTICATED,falta el esquema Bearer,NaN,NaN,NaN,NaN
1,2026-10-09T18:21:21+0000,reemplazo-sintetico,/gout.Demanda/Predecir,NaN,NaN,UNAUTHENTICATED,token vencido,NaN,NaN,NaN,NaN
2,2026-10-09T18:21:21+0000,reemplazo-sintetico,/gout.Demanda/Predecir,NaN,NaN,UNAUTHENTICATED,Signature verification failed,NaN,NaN,NaN,NaN
3,2026-10-09T18:21:21+0000,reemplazo-sintetico,/gout.Demanda/Predecir,visita,invitado,PERMISSION_DENIED,rol sin permiso,NaN,NaN,NaN,NaN
4,2026-10-09T18:21:21+0000,reemplazo-sintetico,/gout.Demanda/Predecir,ana,planificador,INVALID_ARGUMENT,NaN,S03,SKU-0412,8e628779e6a74ee0,0.07
5,2026-10-09T18:21:21+0000,reemplazo-sintetico,/gout.Demanda/Predecir,ana,planificador,OK,NaN,S03,SKU-0412,3d7e86ccfbef3a14,13.09
6,2026-10-09T18:21:21+0000,reemplazo-sintetico,/gout.Demanda/Predecir,airflow-dag,sistema,OK,NaN,S03,SKU-0412,3d7e86ccfbef3a14,1.01


### TODO 3 — De `insecure_port` a TLS

El interceptor resuelve **quién** llama; TLS resuelve **que nadie lea ni modifique** lo que viaja
(incluido el propio JWT, que en un canal sin cifrar se puede robar y reutilizar).

```python
# Servidor
with open("server.key", "rb") as f: clave = f.read()
with open("server.crt", "rb") as f: cert = f.read()
credenciales = grpc.ssl_server_credentials([(clave, cert)])
# mTLS: además exigir certificado del cliente
# credenciales = grpc.ssl_server_credentials([(clave, cert)], root_certificates=ca_clientes,
#                                            require_client_auth=True)
servidor.add_secure_port("0.0.0.0:50051", credenciales)

# Cliente
canal = grpc.secure_channel("gout-api:50051", grpc.ssl_channel_credentials(root_certificates=ca))
```

En el TP final (Docker Compose), los certificados se montan como *secrets* del contenedor y nunca
se suben al repo. Para tráfico interno entre servicios (Airflow → gRPC) conviene **mTLS**, así el servidor
también verifica a quien llama. La celda siguiente lo demuestra con un certificado autofirmado de prueba.

In [ ]:
# Demostración con un certificado autofirmado (solo prueba; en producción lo emite una CA)
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa
import datetime

clave = rsa.generate_private_key(public_exponent=65537, key_size=2048)
nombre = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, "localhost")])
ahora = datetime.datetime.now(datetime.timezone.utc)
cert = (x509.CertificateBuilder().subject_name(nombre).issuer_name(nombre)
        .public_key(clave.public_key()).serial_number(x509.random_serial_number())
        .not_valid_before(ahora).not_valid_after(ahora + datetime.timedelta(days=1))
        .add_extension(x509.SubjectAlternativeName([x509.DNSName("localhost")]), critical=False)
        .sign(clave, hashes.SHA256()))
clave_pem = clave.private_bytes(serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                                serialization.NoEncryption())
cert_pem = cert.public_bytes(serialization.Encoding.PEM)

servidor_tls = crear_servidor()
port_tls = servidor_tls.add_secure_port("localhost:0", grpc.ssl_server_credentials([(clave_pem, cert_pem)]))
servidor_tls.start()

stub_tls = gpbg.DemandaStub(grpc.secure_channel(f"localhost:{port_tls}",
                                                grpc.ssl_channel_credentials(root_certificates=cert_pem)))
r = stub_tls.Predecir(consulta, metadata=bearer(emitir_token("ana", "planificador")))
print(f"TLS + JWT -> {r.unidades:.1f} unidades")

# Un cliente que intenta hablar sin cifrar con el puerto TLS no obtiene respuesta
try:
    gpbg.DemandaStub(grpc.insecure_channel(f"localhost:{port_tls}")).Predecir(
        consulta, metadata=bearer(emitir_token("ana", "planificador")), timeout=2)
except grpc.RpcError as e:
    print("canal sin cifrar contra puerto TLS ->", e.code().name)
_ = servidor_tls.stop(0)

TLS + JWT -> 25.7 unidades
canal sin cifrar contra puerto TLS -> UNAVAILABLE


I1009 18:21:21.957019     358 ssl_transport_security.cc:2530] Handshake failed with error SSL_ERROR_SSL: error:100000f7:SSL routines:OPENSSL_internal:WRONG_VERSION_NUMBER: Invalid certificate verification context


### Gobernanza: model card

Se genera junto al modelo y se versiona con él. Incluye el hash del artefacto que está sirviendo el
servicio, para que cualquiera pueda verificar que el modelo en producción es el documentado.

In [ ]:
model_card = f"""# Model card — Pronóstico de demanda Gout

| Campo | Valor |
|---|---|
| Versión | {MODELO_VERSION} |
| Tipo | {type(MODELO).__name__} (regresión: unidades por sucursal, SKU y día) |
| SHA-256 del artefacto | `{MODELO_SHA256}` |
| Features (en orden) | {", ".join(FEATURES)} |
| Expuesto por | gRPC `gout.Demanda/Predecir`, con JWT + rol, límite de {MAX_MSG // 1024} KB y TLS en producción |

## Uso previsto
Planificar la producción diaria por sucursal y SKU de una cadena de panaderías sin gluten.
Usuarios: planificadores (rol `planificador`) y procesos automáticos como el DAG de Airflow (rol `sistema`).

## Usos fuera de alcance
Decisiones de precios o evaluación de desempeño del personal. Sucursales o SKUs nuevos sin historia
(sin `lag_7` el pronóstico no es confiable).

## Datos
Ventas históricas de la cadena con sucursales anonimizadas. Los datos son comercialmente sensibles:
el servicio no registra features crudas, solo su hash.

## Riesgos y mitigaciones
- Acceso no autorizado al pronóstico → JWT con vencimiento y autorización por rol.
- Robo del token en tránsito → TLS (mTLS entre servicios internos).
- Denegación de servicio con mensajes gigantes → límite de {MAX_MSG // 1024} KB por mensaje.
- Entradas mal formadas → validación de cantidad y rango de features (`INVALID_ARGUMENT`).
- Reemplazo silencioso del modelo → hash SHA-256 publicado en esta card; el servicio no arranca si no coincide con `GOUT_MODELO_SHA256`.
- Trazabilidad → auditoría JSON por llamada en `{AUDIT_PATH}`.

## Limitaciones
Depende de patrones de los años de entrenamiento; feriados atípicos y promociones pueden desviarlo.
Requiere monitoreo de drift (previsto en el TP final).
"""
Path("MODEL_CARD_gout.md").write_text(model_card, encoding="utf-8")
print(model_card)

# Model card — Pronóstico de demanda Gout

| Campo | Valor |
|---|---|
| Versión | reemplazo-sintetico |
| Tipo | HistGradientBoostingRegressor (regresión: unidades por sucursal, SKU y día) |
| SHA-256 del artefacto | `cec467bd76692a18ca2b75477d7309f37c9a02e5e43f3f7d4b2a660a5462bf2c` |
| Features (en orden) | sucursal_cod, sku_cod, dia_semana, mes, es_feriado, lag_1, lag_7, media_movil_7 |
| Expuesto por | gRPC `gout.Demanda/Predecir`, con JWT + rol, límite de 64 KB y TLS en producción |

## Uso previsto
Planificar la producción diaria por sucursal y SKU de una cadena de panaderías sin gluten.
Usuarios: planificadores (rol `planificador`) y procesos automáticos como el DAG de Airflow (rol `sistema`).

## Usos fuera de alcance
Decisiones de precios o evaluación de desempeño del personal. Sucursales o SKUs nuevos sin historia
(sin `lag_7` el pronóstico no es confiable).

## Datos
Ventas históricas de la cadena con sucursales anonimizadas. Los datos son comercialmente sensibles:
el servic

## 4. Reflexión

**1. ¿Qué parte del ataque frena el interceptor y qué parte frena TLS?**
El interceptor frena el **acceso no autorizado**: sin un JWT válido, firmado con nuestro secreto, vigente y con un rol habilitado,
la llamada se corta antes de llegar al modelo (`UNAUTHENTICATED` o `PERMISSION_DENIED`). Pero no protege lo que viaja:
en un canal sin cifrar, alguien en el medio puede leer los pronósticos (información comercial de Gout) o **robar el token y
reutilizarlo** mientras no venza. Eso lo frena **TLS**, que cifra y autentica el canal; con **mTLS**, además el servidor verifica
al cliente. Son complementarios: TLS sin interceptor deja entrar a cualquiera con un canal cifrado, y el interceptor sin TLS
expone el token.

**2. ¿Para qué sirve el límite de tamaño de mensaje?**
Para evitar una **denegación de servicio**: una consulta legítima pesa menos de 100 bytes, y un mensaje de cientos de KB
obligaría al servidor a reservar memoria y deserializar antes de cualquier control. Con el límite de 64 KB, gRPC lo rechaza
con `RESOURCE_EXHAUSTED` **antes** de deserializarlo (por eso ni siquiera aparece en la auditoría).

**3. ¿Qué elemento de SAIF cubre el control principal?**
- **Bases de seguridad sólidas para el ecosistema de IA:** autenticación con JWT, autorización por rol, TLS y validación de
  entrada son controles clásicos de seguridad aplicados al endpoint del modelo.
- **Armonizar los controles a nivel de plataforma:** el mismo esquema de JWT y roles sirve para REST, GraphQL y gRPC del TP
  final, así los tres accesos al modelo tienen la misma política.
- La **auditoría** aporta a **extender la detección y respuesta**: cada intento rechazado queda registrado con quién y por qué.
- La **model card** aporta a **contextualizar los riesgos del sistema de IA en el proceso de negocio**: documenta usos
  previstos, datos sensibles y el hash del modelo que está en producción.